In [2]:
# Import
import random
import numpy as np
import torch
import json
from tqdm import tqdm
from pathlib import Path
import copy
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset
import os
import csv
from transformers import RobertaModel, RobertaTokenizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from collections import Counter, defaultdict, deque
import re
from rank_bm25 import BM25Okapi

device = "cuda" if torch.cuda.is_available() else "cpu"

# Seed for reproductibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Default paths
ROOT = Path("../Amazon_products") # Root Amazon_products directory
TRAIN_DIR = ROOT / "train"
TEST_DIR = ROOT / "test"

TEST_CORPUS_PATH = os.path.join(TEST_DIR, "test_corpus.txt")  # product_id \t text
TRAIN_CORPUS_PATH = os.path.join(TRAIN_DIR, "train_corpus.txt")

CLASS_HIERARCHY_PATH = ROOT / "class_hierarchy.txt" 
CLASS_RELATED_PATH = ROOT / "class_related_keywords.txt" 
CLASS_PATH = ROOT / "classes.txt" 

SUBMISSION_PATH = "../Submission/submission.csv"  # output file

# Constants
NUM_CLASSES = 531  # total number of classes (0–530)

# Loading functions
def load_classic(path):
    """Load doc into {id: text} dictionary."""
    id2text = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t", 1)
            if len(parts) == 2:
                id, text = parts
                id2text[id] = text
    return id2text

def load_multilabel(path):
    """Load multi-label data into {id: [labels]} dictionary -> for class_hierarchy"""
    id2labels = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) == 2:
                pid, label = parts
                pid = int(pid)
                label = int(label)
                if pid not in id2labels:
                    id2labels[pid] = []
                id2labels[pid].append(label)
    return id2labels

def load_class_keywords(path):
    """Load class keywords into {class_name: [keywords]} dictionary."""
    class2keywords = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if ":" not in line: # accept only valid format
                continue
            classname, keywords = line.strip().split(":", 1)
            keyword_list = [kw.strip() for kw in keywords.split(",") if kw.strip()]
            class2keywords[classname] = keyword_list
    return class2keywords

# Extraction
id2text_test = load_classic(TEST_CORPUS_PATH) # id -> text test
id_list_test = list(id2text_test.keys()) # list id test
print(list(id2text_test.items())[:1])
print(len(id_list_test)) #19658

id2text_train = load_classic(TRAIN_CORPUS_PATH) # id -> text train
id_list_train = list(id2text_train.keys()) # list id train
print(list(id2text_train.items())[:1])
print(len(id_list_train)) #29487

id2class = load_classic(CLASS_PATH) # id class -> class text
print(list(id2class.items())[:1])
print(len(id2class)) #531

class2hierarchy = load_multilabel(CLASS_HIERARCHY_PATH) # id parents -> children (taxonomy)
print(list(class2hierarchy.items())[:1])
print(len(class2hierarchy)) #69

class2related = load_class_keywords(CLASS_RELATED_PATH) # id class -> related keywords
print(list(class2related.items())[:1])
print(len(class2related)) #531


c:\Users\noamc\Documents\insa_korea\Cours\big data\final proj\project_release\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[('0', "conair cs15tcs professional straight styles straightening iron woah ! sure this straightener looks like all the other crappy straightners in the world , but there 's a twist to this one ! it is my first straightner and i 've had it for about 7 months . i bought it only because i was desperate for a cheap straightener because my hair is very thick , long , wavy ! i 'm looking for a new straighner right now ... but until then this one is doing just fine . if it works for me , it will work for you !")]
19658
[('0', 'omron hem 790it automatic blood pressure monitor with advanced omron health management software so far this machine has worked well and is very simple to use . it is nice to have immediate feedback on the bloodpressure effects of my various exercises , food consumption , and relaxation or stress levels .')]
29487
[('0', 'grocery_gourmet_food')]
531
[(0, [1, 8, 208, 211, 213, 216, 229, 255, 265, 218, 271, 277, 249, 288, 313, 357])]
69
[('grocery_gourmet_food', ['snacks'

In [3]:
def hierarchy_consistency(silver, hierarchy):
    """Hierarchy consistency in a hierarchy given for our silver labels"""
    ok = 0
    total = 0
    for labels in silver.values():
        L = set(labels)
        for parent, children in hierarchy.items():
            for child in children:
                if child in L:
                    total += 1
                    if parent in L:
                        ok += 1
    return ok / total if total > 0 else 0

def label_coverage(silver_labels, num_classes=531):
    """
    silver_labels : { review_id: [label1, label2, ...] }
    returns coverage_ratio, covered_classes
    """
    covered = set()

    for i, labels in silver_labels.items():
        for lbl in labels:
            if 0 <= lbl < num_classes:
                covered.add(lbl)

    coverage_ratio = len(covered) / num_classes
    return coverage_ratio, sorted(list(covered))

def preprocess_text(text):
    """
    Clean text ONLY for BM25 (NOT for MPNet).
    """
    text = text.lower()
    text = text.replace("_", " ")        
    text = re.sub(r"[>&]", " ", text)
    text = re.sub(r"[^a-z0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def bm25_tokenize(text):
    return preprocess_text(text).split()

In [4]:
def expand_with_hierarchy2(label, tree, label_embeddings):
    """Upgarde Version : Return [label, best_parent, best_grandparent] based on embedding similarity in a DAG."""
    
    # Build child -> parents map
    parent_map = {}
    for parent, children in tree.items():
        for child in children:
            parent_map.setdefault(child, []).append(parent)

    path = [label]

    def best_parent(child):
        parents = parent_map.get(child, [])
        if len(parents) <= 1:
            return parents[0] if parents else None

        child_emb = label_embeddings[child]
        parent_embs = label_embeddings[parents]
        sims = torch.matmul(parent_embs, child_emb)
        return parents[torch.argmax(sims).item()]

    parent = best_parent(label)
    if parent is not None:
        path.append(parent)
        gp = best_parent(parent)
        if gp is not None:
            path.append(gp)

    return path

def expand_with_hierarchy(start_labels, tree):
    """
    Expand a list of core labels by adding ALL their ancestors
    (parents, parents of parents, etc.), recursively.
    """
    # We need to inverse the hierarchy to work on it
    parent_map = {}
    for parent, kids in tree.items():
        for kid in kids:
            if kid not in parent_map:
                parent_map[kid] = []
            parent_map[kid].append(parent)

    collected = set(start_labels)
    keep_going = True

    # Add ancestors to collected
    while keep_going:
        keep_going = False
        for current in list(collected):
            for parent in parent_map.get(current, []):
                if parent not in collected:
                    collected.add(parent)
                    keep_going = True

    # Heuristic: we keep the 3 most precise labels (the lowest in the hierarchy)
    return sorted(collected)[-3:]
    

def propagate_hierarchy_simple(label_embeddings, class_hierarchy, alpha=0.2):
    """
    Propagates parent info into each class embedding.
    Each class gets mixed with the average of its parents.
    Embeddings are normalized after propagation.
    """

    # build reverse mapping child -> list of parents
    child2parents = {}
    for parent, children in class_hierarchy.items():
        for c in children:
            child2parents.setdefault(c, []).append(parent)

    updated = label_embeddings.clone()

    # blend parent info into each class embedding
    for class_id in range(NUM_CLASSES):
        class_id_str = str(class_id)

        # skip if no hierarchy info for this class
        if class_id not in child2parents:
            continue

        # get parent ids
        parents = child2parents[class_id]

        # mix class embedding with the average of its parents
        parent_vec = label_embeddings[parents].mean(dim=0)
        updated[class_id] = (1 - alpha) * label_embeddings[class_id] + alpha * parent_vec

    # normalize embeddings so that similarity depends only on their semantic direction;
    # this avoids being influenced by vector magnitude, which often reflects sentence length
    # rather than meaning.
    norms = torch.norm(updated, dim=1, keepdim=True)
    updated = updated / (norms + 1e-8)

    return updated

def propagate_hierarchy_dag(label_embeddings, class_hierarchy, alpha=0.2):
    """
    Hierarchical propagation: embeddings travel from parents to children 
    through the entire depth of the DAG.
    
    topological sort to ensure parents are always processed first
    updated parent embeddings for propagation
    """
    # build reverse mapping child -> list of parents
    child2parents = {}
    for parent, children in class_hierarchy.items():
        for c in children:
            child2parents.setdefault(c, []).append(parent)

    updated = label_embeddings.clone()

    # Topological sort
    degree = defaultdict(int)
    graph = defaultdict(list)

    # build graph and degree
    for parent, children in class_hierarchy.items():
        for c in children:
            graph[parent].append(c)
            degree[c] += 1

    # nodes with no parents = roots
    queue = deque([n for n in range(NUM_CLASSES) if degree[n] == 0])
    topo_order = []

    # Topological traversal:
    # We repeatedly take nodes whose parents are all already processed (degree == 0).
    # When a node is removed from the queue, we "free" its children by decreasing their
    # remaining-parent count (degree). A child is pushed into the queue only when all
    # its parents have been processed. This ensures parents always appear before children
    # in topo_order.
    while queue:
        node = queue.popleft()
        topo_order.append(node)
        for child in graph[node]:
            degree[child] -= 1
            if degree[child] == 0:
                queue.append(child)

    # blend parent info into each class embedding
    for class_id in topo_order:
        if class_id in child2parents:
            parents = child2parents[class_id]

            # use UPDATED parents (key difference)
            parent_vec = updated[parents].mean(dim=0)

            updated[class_id] = (1 - alpha) * updated[class_id] + alpha * parent_vec

    # normalize embeddings so that similarity depends only on their semantic direction;
    # this avoids being influenced by vector magnitude, which often reflects sentence length
    # rather than meaning.
    norms = torch.norm(updated, dim=1, keepdim=True)
    updated = updated / (norms + 1e-8)

    return updated

In [5]:
def ensure_tensor(x):
    # To avoid problem
    if isinstance(x, torch.Tensor):
        return x
    if isinstance(x, np.ndarray):
        return torch.from_numpy(x)
    if isinstance(x, list):
        return torch.stack(x)
    raise TypeError(f"Unsupported type {type(x)}")

def get_embeddings(texts, model, batch_size=64, save_path=None, force_recompute=False):
    """
    Compute sentence embeddings using a transformer model.
    Can load from a cached file if available, otherwise encodes the texts
    Returns a tensor of shape (N, D).
    """
    # load cache (help from GPT for this one)
    if save_path and os.path.exists(save_path) and not force_recompute:
        print(f"Loading from {save_path}")
        emb = torch.load(save_path, map_location="cpu")
        if isinstance(emb, np.ndarray):
            emb = torch.from_numpy(emb)
        return emb

    # encode with our transformers
    emb = model.encode(texts, batch_size=batch_size, show_progress_bar=True, convert_to_tensor=True)

    # pb of instance (safety check)
    emb = ensure_tensor(emb).cpu()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        torch.save(emb, save_path)
        print(f"Saved to {save_path}")
    return emb

def get_enriched_category_with_hierarchy(class_id, id2class, class2related, class_hierarchy):
    """
    Build an enriched text description for a given class.
    The description includes: the class name (cleaned), the names of its parents,
    and any related keywords linked to the class. This is meant to provide a
    richer textual representation for embedding models.
    """
    class_name = id2class[str(class_id)]
    # We remove the underscores because otherwise it looks odd and the embeds don't understand it as well
    clean_name = class_name.replace('_', ' ')
    
    # Parents
    parents = class_hierarchy.get(str(class_id), {}).get("parents", [])
    parent_names = []
    for p in parents:
        if 0 <= p < NUM_CLASSES:
            parent_name = id2class[str(p)].replace('_', ' ')
            parent_names.append(parent_name)
    
    # Keywords
    keywords = class2related.get(class_name, [])
    
    # Combine
    parts = [clean_name]
    if parent_names:
        parts.extend(parent_names)
    if keywords:
        parts.extend(keywords)

    final = " ".join(parts)
    return final

In [6]:
# load multilingual sentence-transformer model -> model for our embeddings
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)
model = model.to(device)

In [7]:
# Normalize both sets of scores
def rank_normalize(x):
    ranks = x.argsort().argsort().float()
    return 2 * (ranks / (len(x)-1)) - 1   # range = [-1, 1]

def generate_silver_labels(train_texts, train_ids, test_texts, test_ids, id2class, class2related, model, class_hierarchy,
    output_path_train="Silver/silver_train_mini25.json",
):
    """ Main Function :
    Generate silver labels for train and test data using semantic similarity.
    The function builds enriched category descriptions, encodes them,
    applies hierarchical propagation, computes similarities between all reviews
    and all category embeddings, expands predictions through the hierarchy,
    and saves both hierarchical and non-hierarchical silver labels.
    
    Returns:
        (silver_train, silver_train_nohier)
        where each dict maps product_id -> {labels}.
    """

    save_label_nohier = "Embeddings/labels_base_mini25.pt"
    save_label_hier = "Embeddings/labels_mini25.pt"
    save_corpus = "Embeddings/X_train_test_mini25.pt"

    # Creation of labels embeddings with hierarchy and without hierarchy propagation
    enriched_categories = [
        get_enriched_category_with_hierarchy(i, id2class, class2related, class_hierarchy)
        for i in tqdm(range(NUM_CLASSES), desc="Enriching")
    ]

    category_corpus_bm25 = [bm25_tokenize(t) for t in enriched_categories]
    bm25 = BM25Okapi(category_corpus_bm25)

    base_category_embeddings = get_embeddings(
        enriched_categories,
        model=model,
        batch_size=64,
        save_path=save_label_nohier,
        force_recompute=True
    )
    # Ensure tensor format
    base_category_embeddings = ensure_tensor(base_category_embeddings)

    hierarchical_embeddings = propagate_hierarchy_dag(
        label_embeddings=base_category_embeddings,
        class_hierarchy=class_hierarchy,
        alpha=0.2,
    )
    torch.save(hierarchical_embeddings, save_label_hier)
    
    # Creation of doc embeddings
    # Concatenation -> help for generalization    
    all_texts = train_texts + test_texts
    all_texts_bm25 = [preprocess_text(t) for t in all_texts]

    all_ids = train_ids + test_ids

    review_embeddings = get_embeddings(
        all_texts,
        model=model,
        batch_size=64,
        save_path=save_corpus,
        force_recompute=True
    )

    # Ensure both are tensors (check)
    review_embeddings = ensure_tensor(review_embeddings)
    hierarchical_embeddings = ensure_tensor(hierarchical_embeddings)

    # Move to device for computation
    review_embeddings = review_embeddings.to(device)
    hierarchical_embeddings = hierarchical_embeddings.to(device)

    # Compute similarity on device (innerproduct on normalized embeddings => cos similarity) -> hierarchical similarity
    all_similarities = torch.matmul(review_embeddings, hierarchical_embeddings.T)
    all_similarities = all_similarities.cpu()

    all_similarities2 = torch.matmul(review_embeddings,base_category_embeddings.to(device).T)
    all_similarities2 = all_similarities2.cpu()

    silver_train = {}
    silver_train_nohier = {}

    n_train = len(train_ids)

    for idx, rid in enumerate(tqdm(all_ids, desc="Assigning")):

        # Bm25
        doc_tokens = bm25_tokenize(all_texts_bm25[idx])
        bm25_scores = bm25.get_scores(doc_tokens)
        bm25_scores = torch.tensor(bm25_scores, dtype=torch.float32)

        alpha = 0.3   # weight between MPNet and BM25
        sims_mpnet1 = all_similarities[idx]
        sims_mpnet2 = all_similarities2[idx]
        sims_bm25  = bm25_scores
        
        s_mp = rank_normalize(sims_mpnet1)
        s_bm = rank_normalize(sims_bm25)

        sims = alpha * s_bm + (1 - alpha) * s_mp

        # keep only the best match; simple heuristic: we take top-1 and expand it through the hierarchy
        # (we can do this because we are generating labels here, not training a model)        
        topk_scores, topk_idx = torch.topk(sims, k=1) 
        topk_idx = topk_idx.tolist()
        expanded = expand_with_hierarchy2(
            topk_idx[0],
            class_hierarchy,
            hierarchical_embeddings
        )
        expanded_scores = [float(sims[l]) for l in expanded]

        pairs = sorted(zip(expanded, expanded_scores), key=lambda t: t[1], reverse=True)  # sort labels by descending score
        sorted_labels, sorted_scores = zip(*pairs)  # separate sorted labels and scores

        final_labels = sorted_labels

        record = {
            "labels": final_labels,
        }

        if idx < n_train: # because there is also test embeddings in all_ids so we need to care
            silver_train[rid] = record

        s_mp2 = rank_normalize(sims_mpnet2)
        sims2 = alpha * s_bm + (1 - alpha) * s_mp2

        topk_scores2, topk_idx2 = torch.topk(sims2, k=1)
        topk_idx2 = topk_idx2.tolist()
        expanded2 = expand_with_hierarchy2(
            topk_idx2[0],
            class_hierarchy,
            base_category_embeddings
        )
        expanded_scores2 = [float(sims2[l]) for l in expanded2]

        pairs = sorted(zip(expanded2, expanded_scores2), key=lambda t: t[1], reverse=True)  # sort labels by descending score
        sorted_labels2, sorted_scores2 = zip(*pairs)  # separate sorted labels and scores

        final_labels2 = sorted_labels2

        record_nohier = {
            "labels": sorted_labels2,
        }

        if idx < n_train:
            silver_train_nohier[rid] = record_nohier

    # Creation SILVER
    os.makedirs("Silver", exist_ok=True)
    json.dump(silver_train, open(output_path_train, "w", encoding="utf-8"), indent=2, ensure_ascii=False)

    output_path_train_nohier = output_path_train.replace(".json", "_nohier.json")
    json.dump(silver_train_nohier,open(output_path_train_nohier, "w", encoding="utf-8"),indent=2, ensure_ascii=False)

    return silver_train, silver_train_nohier


In [8]:
# Exec
print("Generating silver labels")

silver_train_safe, silver_train_safe_nohier = generate_silver_labels(
    list(id2text_train.values()),
    id_list_train,
    list(id2text_test.values()),
    id_list_test,
    id2class,
    class2related,
    model,
    class2hierarchy,
    output_path_train="Silver/silver_train_mini25.json",
)

# Constructions of dict for silvers labels
silver_train_labels = {
    pid: info["labels"]
    for pid, info in silver_train_safe.items()
}

silver_train_labels_nohier = {
    pid: info["labels"]
    for pid, info in silver_train_safe_nohier.items()
}

# Some statistics
# Hierarchical
consistency = hierarchy_consistency(silver_train_labels, class2hierarchy)
print(f"\nHierarchy Consistency: {consistency:.2%}")
coverage, classes = label_coverage(silver_train_labels)
print(f"Coverage: {coverage:.2%}")
print(f"Covered classes: {len(classes)}/{NUM_CLASSES}")

# No Hier
consistency = hierarchy_consistency(silver_train_labels_nohier, class2hierarchy)
print(f"\nHierarchy Consistency: {consistency:.2%}")
coverage, classes = label_coverage(silver_train_labels_nohier)
print(f"Coverage: {coverage:.2%}")
print(f"Covered classes: {len(classes)}/{NUM_CLASSES}")

Generating silver labels


Batches: 100%|██████████| 9/9 [00:00<00:00, 24.12it/s]


Saved to Embeddings/labels_base_mini25.pt


Batches: 100%|██████████| 768/768 [00:44<00:00, 17.41it/s]


Saved to Embeddings/X_train_test_mini25.pt


Assigning: 100%|██████████| 49145/49145 [04:37<00:00, 176.87it/s]



Hierarchy Consistency: 93.17%
Coverage: 99.44%
Covered classes: 528/531

Hierarchy Consistency: 93.16%
Coverage: 99.44%
Covered classes: 528/531
